# 04. YOLO11 (одностадійний, без anchors)

У YOLO11 (Ultralytics) anchors немає: кожна позиція сітки P3/P4/P5 (stride 8/16/32) сама передбачає відстані до чотирьох сторін боксу (DFL). Позитиви вибирає TaskAlignedAssigner: top-k позицій за добутком score класу й IoU, центр яких лежить усередині GT.

Протокол: ті самі кадри з `prepared/` (960 px, ignore замальовані), ті самі val-кадри, оцінка `common.evaluate` в оригінальних координатах.
Навчання: ваги COCO `yolo11s.pt`, 25 епох, стандартний рецепт Ultralytics (SGD/AdamW auto, mosaic до останніх 5 епох, фліп, HSV і scale), seed 42.
Швидкість: batch 1, препроцесинг + інференс + NMS.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ostapmn/ucu_ai_hw_2.git"
if Path("/kaggle/working").exists() and not Path("common.py").exists():
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    os.chdir(repo / "code")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, ".")
DEBUG = os.environ.get("HW2_DEBUG") == "1"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

import analysis as an
import common as c
import yolo_utils as yu
from ultralytics import YOLO

USE_WANDB = False
yu.setup(wandb=USE_WANDB)
c.set_seed()
P = c.paths()
P1280 = c.paths("prepared_1280")
OUT = P.out / "anchor_free"
print("відновлено з Input:", c.restore_from_inputs(P, "anchor_free"))
GT = P.prep / "val" / "annotations_eval.json"
RAW = P.data / "val" / "annotations_coco.json"
BASE = dict(epochs=25, batch=16, workers=4, close_mosaic=5, deterministic=False, plots=True,
            device=yu.gpu_devices())
if DEBUG:
    BASE.update(epochs=1, batch=4, workers=0, plots=False, device="cpu")
DATA = {"960": yu.write_data_yaml(P.prep, OUT / "data_960.yaml", train_every=20 if DEBUG else 1)}
if (P1280.prep / "val" / "annotations_eval.json").exists():
    DATA["1280"] = yu.write_data_yaml(P1280.prep, OUT / "data_1280.yaml", train_every=20 if DEBUG else 1)
P2_YAML = yu.write_yolo11_p2_yaml(OUT / "yolo11s-p2.yaml")
print(BASE, list(DATA))

## 1. Експерименти і простір пошуку

Базова модель і три окремі зміни: додаткова голова P2 (stride 4) для високої роздільності, вхід 1280 px і кількість кандидатів у TaskAlignedAssigner.

In [ ]:
EXPERIMENTS = {
    "yolo_base": dict(model=None, data="960", imgsz=960, tal_topk=10),
    "yolo_p2": dict(model=P2_YAML, data="960", imgsz=960, tal_topk=10),
    "yolo_1280": dict(model=None, data="1280", imgsz=1280, tal_topk=10, batch=8),
    "yolo_tal13": dict(model=None, data="960", imgsz=960, tal_topk=13),
}
RUN = ["yolo_base", "yolo_p2", "yolo_1280", "yolo_tal13"]
display(pd.DataFrame(EXPERIMENTS).T.drop(columns="model").assign(head=lambda d: ["P2-P5" if EXPERIMENTS[n]["model"] else "P3-P5" for n in d.index]))


def prep_of(cfg):
    return P1280.prep if cfg["data"] == "1280" else P.prep


def run_experiment(name):
    cfg = EXPERIMENTS[name]
    if cfg["data"] not in DATA:
        print(f"{name}: немає prepared_{cfg['data']}, пропускаю")
        return None
    args = {**BASE, "imgsz": 320 if DEBUG else cfg["imgsz"], "batch": 4 if DEBUG else cfg.get("batch", BASE["batch"])}
    if cfg["tal_topk"] != 10:
        yu.patch_tal_topk(cfg["tal_topk"])
        args["device"] = 0 if torch.cuda.is_available() else "cpu"
    run = yu.train(cfg["model"], "yolo11s.pt", DATA[cfg["data"]], OUT, name, **args)
    yu.patch_tal_topk(10)
    model = YOLO(run / "weights" / "best.pt")
    dets, _ = yu.predict_prepared(model, prep_of(cfg), "val", imgsz=args["imgsz"])
    c.save_json(dets, OUT / name / "val_dets.json")
    metrics = c.evaluate(GT, dets, per_class=True)
    ms = yu.measure_speed(model, prep_of(cfg), imgsz=args["imgsz"], n=5 if DEBUG else 100)
    hours = float(yu.results_csv(run)["time"].iloc[-1]) / 3600
    ngpu = len(args["device"]) if isinstance(args["device"], list) else 1
    return an.log_result(P.results, name, "one-stage anchor-free", metrics, ms_per_img=ms, epochs=args["epochs"],
                         wall_hours=hours, gpu_hours=hours * ngpu, imgsz=args["imgsz"], config=str(cfg))


for name in RUN:
    print("=" * 20, name)
    run_experiment(name)
    torch.cuda.empty_cache()

## 2. Результати за розміром об'єктів і порівняння з anchor-based

In [ ]:
COLS = ["family", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR_max", "ms_per_img", "gpu_hours"]
res = an.all_results(P).set_index("name")
display(res[COLS].round(3))
mine = res[res.family == "one-stage anchor-free"]
display(mine[[f"AP_{k}" for k in c.CLASSES]].round(3))
BEST = mine.mAP.idxmax()
print("найкраща конфігурація:", BEST)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for name in RUN:
    if not (OUT / name / "results.csv").exists():
        continue
    r = yu.results_csv(OUT / name)
    axes[0].plot(r.epoch, r["train/box_loss"], label=name)
    axes[1].plot(r.epoch, r["metrics/mAP50-95(B)"], label=name)
    axes[2].plot(r.epoch, r["metrics/recall(B)"], label=name)
for ax, t in zip(axes, ["box loss (train)", "val mAP50-95 (Ultralytics, без ignore)", "val recall (Ultralytics)"]):
    ax.set_title(t, fontsize=10)
    ax.set_xlabel("епоха")
axes[1].legend(fontsize=8)
fig.savefig(P.images / "yolo_curves.png", bbox_inches="tight", dpi=130)
plt.show()

### Recall за розміром об'єкта (поріг score 0.3)

In [ ]:
recall = {}
for name in RUN:
    path = OUT / name / "val_dets.json"
    if path.exists():
        _, g_df, _ = an.error_breakdown(GT, c.load_json(path), score_thr=0.3)
        recall[name] = g_df.groupby("size").found.mean()
for fam, sub in (("two-stage", "two_stage"), ("one-stage anchor-based", "one_stage_anchor")):
    fam_res = res[res.family == fam]
    path = an.find_run_file(P, f"{sub}/{fam_res.mAP.idxmax()}/val_dets.json") if len(fam_res) else None
    if path is not None:
        _, g_df, _ = an.error_breakdown(GT, c.load_json(path), score_thr=0.3)
        recall[path.parent.name] = g_df.groupby("size").found.mean()
pd.DataFrame(recall).T.reindex(columns=["small", "medium", "large"]).round(3)

## 3. Помилки найкращої моделі

In [ ]:
dets = c.load_json(OUT / BEST / "val_dets.json")
d_df, g_df = an.error_report(GT, dets, P.data / "val", RAW, P.images, prefix="yolo", score_thr=0.3)

## Висновки

Усі числа з таблиць цього ноутбука (val, `common.evaluate`), по одному запуску на конфігурацію. Порівняння з anchor-based бере результати `02` і `03`, де інші рецепти навчання (6 епох torchvision проти 25 епох Ultralytics з mosaic), тому це порівняння налаштованих моделей, а не чистих архітектур.

**Призначення позицій без anchors.** TaskAlignedAssigner бере top-k позицій за добутком score класу й IoU, центр яких лежить усередині GT. Малий об'єкт на stride 8 (P3) покриває мало таких центрів, тому кандидатів для нього мало. Додаткова голова P2 (stride 4) збільшує їх число. У щільних сценах сусідні об'єкти конкурують за ті самі позиції.

**Простір пошуку та результати.**
- yolo_base: mAP 0.171, AP50 0.338, AP_small 0.090, 16.9 мс.
- yolo_p2: 0.179, 0.361, 0.097, 24.1 мс, 3.4 GPU-год.
- yolo_tal13 (top-k 13 замість 10): 0.181, 0.345, 0.093, 17.0 мс, 1.7 GPU-год.

P2 піднімає AP50 (+0.023), AP_small і дрібні bicycle (0.135 → 0.183) та bus (0.025 → 0.108), але коштує +42% часу інференсу й +63% GPU-годин, а AP_large трохи падає (0.309 → 0.298). tal13 безкоштовний за швидкістю й дає найкращий mAP, AP75 (0.164) і AP_large (0.342, +0.033), але bus падає до 0.005 (у val лише 132 об'єкти). `yolo_1280` не виконався: немає датасету `prepared_1280`.

**Recall за розміром (score ≥ 0.3).** base 0.398 / 0.568 / 0.755, p2 0.311 / 0.524 / 0.709, tal13 0.375 / 0.522 / 0.779 для малих, середніх і великих. P2 має вищий AP_small, але нижчий recall при фіксованому порозі. Імовірно, score на P2 нижчі, і частину знайдених малих відсікає поріг 0.3, а AP враховує їх на всіх порогах.

**Порівняння з anchor-based (за розмірами).**
- AP_small: YOLO 0.090, 0.097, 0.093; Faster R-CNN 0.062 і 0.077; RetinaNet 0.032 і 0.052.
- AP_large: YOLO 0.309, 0.298, 0.342; Faster R-CNN 0.346 і 0.345; RetinaNet 0.325 і 0.275.
- mAP: найкращий YOLO 0.181, Faster R-CNN 0.175, RetinaNet 0.147.

Anchor-free YOLO кращий на малих об'єктах і однаково добрий на великих, а в 6 разів швидший за обидві anchor-based моделі (17 мс проти 97 і 105 мс).

**Помилки tal13 (score ≥ 0.3):** tp 58.1%, cls 11.2%, loc 6.8%, bg 9.4%, ignore 8.7%, dup 3.3%. Recall: малі 0.375, великі 0.779; bus 0.03, bicycle 0.14, motor 0.50, person 0.58; без перекриття 0.716, часткове 0.416, сильне 0.170.